# Bitwise Operations

Copyright 2024 Google LLC. Adapted under Apache-2.0 from the locked Qualtran v0.7.0 bitwise notebook, commit `bcb17bb25360ed68ce3bc084645db3e929799389`.


In [ ]:
from collections import Counter
from collections.abc import Callable
from typing import cast

import torch
from IPython import display as notebook_display
from sympy import Symbol

from examples.algorithms.addition_catalog import addition_signature_svg
from examples.algorithms.binary_field_arithmetic import FieldOperation
from flagquantum import Circuit
from flagquantum.compiler._component_trace.arithmetic_call import ArithmeticCall
from flagquantum.compiler._component_trace.call_graph import (
    SymbolicCallGraph,
    symbolic_call_graph,
)
from flagquantum.compiler._component_trace.resource_generalization import (
    CliffordCall,
    OnEachCall,
    ResourceCall,
)
from flagquantum.compiler.registers.arithmetic.arithmetic_declarations import (
    arithmetic_call_graph,
)
from flagquantum.compiler.registers.bits.bitwise import bitwise_not_ir, xor_ir
from flagquantum.core.ir.program.register_types import (
    opaque_register_type,
    unsigned_register_type,
)
from flagquantum.core.ir.program.registers import Register
from flagquantum.core.ir.program.signatures import Signature
from flagquantum.core.value_math import SymbolicInt
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg

# IPython exposes untyped presentation functions at this optional display boundary.
SVG = cast(Callable[[bytes], object], notebook_display.SVG)
display = cast(Callable[[object], None], notebook_display.display)


def signature_svg(*items: tuple[str, tuple[ArithmeticCall, Signature]]) -> bytes:
    return addition_signature_svg(
        [(name, call, signature) for name, (call, signature) in items]
    )


def graph(call: ArithmeticCall) -> SymbolicCallGraph[ResourceCall]:
    return symbolic_call_graph(call, arithmetic_call_graph(call), max_depth=1)


def call_title(call: ResourceCall) -> str:
    if isinstance(call, ArithmeticCall):
        label = "Register XOR" if call.operation == "xor" else "Register complement"
        return f"{label} ({call.widths[0]} bits)"
    if isinstance(call, CliffordCall):
        return call.opcode.upper()
    if isinstance(call, OnEachCall):
        return f"Apply X to {call.width} bits"
    return str(call)


def show_graph(value: SymbolicCallGraph[ResourceCall]) -> None:
    display(
        SVG(
            render_call_graph_svg(
                export_call_graph_dot(value.root, value.direct_calls, title=call_title)
            )
        )
    )
    display(value.leaf_counts)


## Register XOR
Xor the value of one register into another via CNOTs.

Maps basis states $|x, y\rangle$ to $|x, y \oplus x\rangle$.

When both registers are in computational basis and the destination is 0,
effectively copies the value of the source into the destination.

#### Parameters
 - `dtype`: Data type of the input registers `x` and `y`. 

#### Registers
 - `x`: The source register.
 - `y`: The target register.


In [ ]:
# Opaque word metadata and word-width resource identity have separate owners.
def xor_example(width: SymbolicInt) -> tuple[ArithmeticCall, Signature]:
    signature = Signature(
        tuple(Register(name, opaque_register_type(width)) for name in ("x", "y"))
    )
    return ArithmeticCall("xor", (width,)), signature


### Example Instances

In [ ]:
xor = xor_example(4)


In [ ]:
xor_symbolic = xor_example(Symbol("n"))


#### Graphical Signature

In [ ]:
display(SVG(signature_svg(("xor", xor), ("xor_symbolic", xor_symbolic))))


### Call Graph

In [ ]:
xor_graph = graph(xor[0])
show_graph(xor_graph)


## Register complement
Flips every bit of the input register.

#### Parameters
 - `dtype`: Data type of the input register `x`. 

#### Registers
 - `x`: A quantum register of type the declared dtype.


In [ ]:
def not_example(width: SymbolicInt) -> tuple[ArithmeticCall, Signature]:
    signature = Signature((Register("x", unsigned_register_type(width)),))
    return ArithmeticCall("bitwise_not", (width,)), signature


### Example Instances

In [ ]:
bitwise_not = not_example(4)


In [ ]:
n = Symbol("n")
bitwise_not_symbolic = not_example(n)


#### Graphical Signature

In [ ]:
display(
    SVG(
        signature_svg(
            ("bitwise_not", bitwise_not), ("bitwise_not_symbolic", bitwise_not_symbolic)
        )
    )
)


### Call Graph

In [ ]:
bitwise_not_graph = graph(bitwise_not[0])
show_graph(bitwise_not_graph)


## Coherent execution and resource interpretation
The signatures preserve opaque versus unsigned words. Resource calls use the existing word-width identity. Symbolic metadata and estimates do not allocate a physical circuit. The four-bit XOR emits four CX and complement emits four X, with zero work, measurements or T gates. The complement graph retains its OnEach child rather than substituting a primitive count for a declared call.


In [ ]:
word = tuple(range(4))
target = tuple(range(4, 8))
gates = xor_ir(word, target)
coefficients = torch.arange(512, dtype=torch.float64)
initial = torch.complex(coefficients + 1, 3 * coefficients - 5)
initial /= initial.norm()
labels = torch.arange(512, dtype=torch.int64)
expected = torch.zeros_like(initial)
expected[labels ^ ((labels >> 5) << 1)] = initial
circuit = Circuit(9, inputs=initial, dtype=torch.complex128)
operation = FieldOperation(9, gates)
circuit.append(operation)
torch.testing.assert_close(circuit.state().flatten(), expected, rtol=0, atol=1e-12)
circuit.append(operation)
torch.testing.assert_close(circuit.state().flatten(), initial, rtol=0, atol=1e-12)
measured_primitives = {
    "xor": dict(Counter(g.name for g in gates)),
    "complement": dict(Counter(g.name for g in bitwise_not_ir(word))),
}
display(measured_primitives)
